# Nemotron-MT LoRA training (Colab, one-click)

**Before Run All:**
1. Runtime -> Change runtime type -> **A100 GPU** (needs Colab Pro/Pro+).
2. Accept the model license once: https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
3. Create a HF token (read scope): https://huggingface.co/settings/tokens

Then **Runtime -> Run all**. When the HF login box appears, paste your token.
The last cell downloads `submission.zip` -> submit it on Kaggle (account muningan).

> Note: Colab's A100 is 40GB, so this uses 4-bit QLoRA. If the model OOMs or
> bitsandbytes can't quantize the MoE experts, use the bf16 fallback
> (train_modal.py on Modal, or the Kaggle kernel) instead.

## 1. Install dependencies (~10 min; compiles Mamba kernels)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!pip install -q -U "transformers>=4.56.2" "peft>=0.15.0" "trl>=0.12.0" accelerate datasets bitsandbytes hf_transfer einops sentencepiece
# Mamba / causal-conv1d CUDA kernels (the hybrid model needs these). Tries a
# prebuilt wheel first; falls back to source compile against Colab's torch.
!pip install -q causal-conv1d
!pip install -q mamba-ssm
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "|",
      round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
try:
    import mamba_ssm, causal_conv1d
    print("mamba_ssm", mamba_ssm.__version__, "causal_conv1d", causal_conv1d.__version__, "OK")
except Exception as e:
    print("WARNING: mamba kernels not importable:", e)

## 2. Hugging Face login (paste your token when prompted)

In [ ]:
from huggingface_hub import login
login()

## 3. Download the training data

In [ ]:
!wget -q "https://raw.githubusercontent.com/everest-an/nemotron-mt-data/master/sft_train.jsonl" -O sft_train.jsonl
import json
rows = [json.loads(l) for l in open("sft_train.jsonl", encoding="utf-8")]
print("rows:", len(rows))
print("sample assistant:\n", rows[0]["messages"][1]["content"][:300])

## 4. Load model in 4-bit + attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

HF_MODEL = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
LORA_RANK = 32
MAXLEN = 2048

tok = AutoTokenizer.from_pretrained(HF_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL, quantization_config=bnb, device_map="auto",
    trust_remote_code=True, attn_implementation="eager", dtype=torch.bfloat16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_RANK, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=r".*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj|up_proj|down_proj)$",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("mem after load:", round(torch.cuda.memory_allocated()/1e9,1), "GB")

## 5. Train (~1-2h on A100)

In [ ]:
import random
from transformers import TrainingArguments, Trainer

random.seed(0); random.shuffle(rows)

def _ids(x):
    if hasattr(x, "keys") and "input_ids" in x: x = x["input_ids"]
    if hasattr(x, "tolist"): x = x.tolist()
    if len(x) and isinstance(x[0], (list, tuple)): x = x[0]
    return [int(t) for t in x]

def encode(rec):
    msgs = rec["messages"]
    p = _ids(tok.apply_chat_template(msgs[:1], tokenize=True, add_generation_prompt=True))
    a = _ids(tok(msgs[1]["content"], add_special_tokens=False)["input_ids"])
    eos = [int(tok.eos_token_id)] if tok.eos_token_id is not None else []
    ids = (p + a + eos)[:MAXLEN]; lab = ([-100]*len(p) + a + eos)[:MAXLEN]
    return {"input_ids": ids, "labels": lab, "attention_mask": [1]*len(ids)}

enc = [encode(r) for r in rows]
enc = [e for e in enc if any(l != -100 for l in e["labels"])]
print("encoded:", len(enc))

class Collator:
    def __call__(self, batch):
        pad = tok.pad_token_id or 0
        m = max(len(b["input_ids"]) for b in batch)
        ids, lab, att = [], [], []
        for b in batch:
            n = m - len(b["input_ids"])
            ids.append(b["input_ids"] + [pad]*n)
            lab.append(b["labels"] + [-100]*n)
            att.append(b["attention_mask"] + [0]*n)
        return {"input_ids": torch.tensor(ids), "labels": torch.tensor(lab),
                "attention_mask": torch.tensor(att)}

args = TrainingArguments(
    output_dir="ckpt",
    per_device_train_batch_size=1, gradient_accumulation_steps=32,
    learning_rate=2e-4, warmup_steps=0, lr_scheduler_type="linear",
    adam_beta1=0.9, adam_beta2=0.95, weight_decay=0.0, max_grad_norm=1e9,
    logging_steps=10, max_steps=600, save_strategy="no", bf16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_pin_memory=False, remove_unused_columns=False, report_to="none",
)
Trainer(model=model, args=args, train_dataset=enc, data_collator=Collator()).train()
print("training done")

## 6. Package + download submission.zip

In [ ]:
import json, os, zipfile
adir = "adapter"; model.save_pretrained(adir)
cfgp = os.path.join(adir, "adapter_config.json")
cfg = json.load(open(cfgp))
cfg["base_model_name_or_path"] = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
cfg["inference_mode"] = True
json.dump(cfg, open(cfgp, "w"), indent=2)
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(adir, "adapter_config.json"), "adapter_config.json")
    z.write(os.path.join(adir, "adapter_model.safetensors"), "adapter_model.safetensors")
print("submission.zip:", round(os.path.getsize("submission.zip")/1e6,1), "MB")
from google.colab import files
files.download("submission.zip")